# 🧬 Model Pipeline – ERD/ERS Dataset (Subject-Dependent)
**File:** `erd_ers_band_power_xlsx_-_ERD_ERS.csv`  
**Task:** 9-class EEG Scenario Classification  
**Protocol:** **Subject-Dependent** – train and test share subjects (stratified split)  
**Output folder:** `erd_ers_subjectdep_outputs/`

> We first pivot the long-format ERD/ERS data into a wide feature matrix,  
> then use a stratified 80/20 split for subject-dependent evaluation.


## 0. Setup

In [1]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

OUTPUT_DIR = "erd_ers_subjectdep_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
SCENARIO_LABELS = {
    0:"Lift L.Hand", 1:"Lift R.Hand", 2:"Lift L.Leg",  3:"Lift R.Leg",
    4:"Open Mouth",  5:"Nod Head",    6:"Shake Head",   7:"Want Water", 8:"Use Bathroom"
}
LABELS_S = [f"S{i+1}" for i in range(9)]
CMAPS    = ['Blues','Greens','Oranges','Reds','Purples','YlOrBr','GnBu','RdPu','BuPu']
print("Setup complete.")


Setup complete.


## 1. Load & Pivot to Wide Format

In [2]:
PATH = "/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/features.csv/erd_ers_band_power.csv"
df   = pd.read_csv(PATH)
print(f"Raw shape: {df.shape}")
print(f"Subjects : {df['subject'].nunique()}")
print(f"Channels : {df['channel'].unique()}")
print(f"Subbands : {df['subband'].unique()}")
print(f"Tasks    : {df['task'].unique()}")
print(f"NaN      : {df.isnull().sum().sum()}")
print(f"Zeros    : {(df.select_dtypes('number') == 0).sum().sum()}")


Raw shape: (14202, 11)
Subjects : 176
Channels : <ArrowStringArray>
['Cz', 'C3', 'C4', ' ']
Length: 4, dtype: str
Subbands : <ArrowStringArray>
['Mu', 'Low_Beta', 'High_Beta']
Length: 3, dtype: str
Tasks    : <ArrowStringArray>
['Thinking']
Length: 1, dtype: str
NaN      : 0
Zeros    : 411


In [3]:
# Extract scenario number and clean empty channel
df['scenario_num'] = df['scenario'].str.extract(r'(\d+)').astype(int)
df = df[df['channel'].str.strip() != ''].copy()
print(f"After cleaning: {df.shape}")

# Pivot: rows = (subject × scenario), cols = metric_task_channel_subband
df['col_key'] = (df['task'] + '_' + df['channel'] + '_' +
                 df['subband'] + '_' + df['time'].fillna('t1'))

pivot = df.pivot_table(
    index=['subject', 'scenario_num'],
    columns='col_key',
    values=['baseline_power (µ)', 'task_power (µ)', 'erd_ers_pct'],
    aggfunc='mean'
).reset_index()

pivot.columns = ['_'.join(filter(None, map(str, c))).strip('_')
                 if isinstance(c, tuple) else c
                 for c in pivot.columns]

print(f"Pivoted: {pivot.shape}")


After cleaning: (14201, 12)
Pivoted: (1578, 29)


In [4]:
id_cols      = ['subject', 'scenario_num']
feature_cols = [c for c in pivot.columns if c not in id_cols]

X_raw = pivot[feature_cols].values.astype(float)
y     = pivot['scenario_num'].values - 1    # 0-indexed

imputer = SimpleImputer(strategy='median')  # NaN → median; zeros kept
X = imputer.fit_transform(X_raw)

print(f"Feature matrix : {X.shape}")
print(f"Zeros preserved: {(X == 0).sum()}")
print(f"NaN remaining  : {np.isnan(X).sum()}")
print(f"Class balance  :\n{pd.Series(y).value_counts().sort_index().to_string()}")


Feature matrix : (1578, 27)
Zeros preserved: 411
NaN remaining  : 0
Class balance  :
0    175
1    176
2    173
3    175
4    176
5    176
6    176
7    175
8    176


## 2. Stratified Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print("Subject-dependent: same subjects appear in both train and test splits.")


## 3. Define Model Pipelines

In [ ]:
models = {
    "Random Forest": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=300, max_depth=15,
                                       random_state=42, n_jobs=-1))
    ]),
    "Logistic Regression": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs',
                                   random_state=42))
    ]),
    "KNN": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=7, n_jobs=-1))
    ]),
    "XGBoost": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                              subsample=0.8, colsample_bytree=0.8,
                              eval_metric='mlogloss', random_state=42,
                              n_jobs=-1, tree_method='hist'))
    ]),
    "LightGBM": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.1,
                               random_state=42, n_jobs=-1, verbose=-1))
    ]),
    "AdaBoost": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', AdaBoostClassifier(n_estimators=150, learning_rate=0.5,
                                   random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ('sc',  StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                           learning_rate=0.1, random_state=42))
    ]),
}

stk_estimators = [
    ('rf',   RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)),
    ('xgb',  XGBClassifier(n_estimators=150, tree_method='hist',
                            eval_metric='mlogloss', random_state=42, n_jobs=-1)),
    ('lgbm', LGBMClassifier(n_estimators=150, random_state=42, verbose=-1)),
]
models["Stacking (RF+XGB+LGBM)"] = Pipeline([
    ('sc',  StandardScaler()),
    ('clf', StackingClassifier(
        estimators=stk_estimators,
        final_estimator=LogisticRegression(max_iter=2000, random_state=42),
        cv=5, n_jobs=-1
    ))
])
print(f"Models: {list(models.keys())}")


## 4. Stratified K-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_results = {}

print("10-Fold Stratified CV (subject-dependent)...\n")
for name, pipe in models.items():
    if "Stacking" in name:
        print(f"  {name:38s}: [skipped]")
        continue
    scores = cross_val_score(pipe, X_train, y_train, cv=skf,
                             scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f"  {name:38s}: {scores.mean():.4f} ± {scores.std():.4f}")


## 5. Train & Evaluate

In [ ]:
test_results = {}
for name, pipe in models.items():
    print(f"  Fitting {name} ...", end='  ', flush=True)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    test_results[name] = {'accuracy': acc, 'f1_weighted': f1, 'y_pred': y_pred}
    print(f"Acc={acc:.4f}  F1={f1:.4f}")


## 6. Visualisations

In [ ]:
# FIG 1 – CV
cv_names = list(cv_results.keys())
cv_means = [cv_results[n].mean() for n in cv_names]
cv_stds  = [cv_results[n].std()  for n in cv_names]
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.barh(cv_names, cv_means, xerr=cv_stds, height=0.5,
               color=plt.cm.Set2(np.linspace(0, 1, len(cv_names))),
               alpha=0.85, error_kw=dict(ecolor='gray', capsize=4))
for bar, v in zip(bars, cv_means):
    ax.text(v+0.003, bar.get_y()+bar.get_height()/2, f'{v:.4f}', va='center', fontsize=9)
ax.axvline(1/9, color='red', ls='--', alpha=0.5, label='Chance (11.1%)')
ax.set_xlabel('Accuracy'); ax.set_xlim(0, 1.1)
ax.set_title("10-Fold Stratified CV – ERD/ERS (Subject-Dependent)",
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_cv_stratified.png", bbox_inches='tight')
plt.show()


In [ ]:
# FIG 2 – Test performance
nt = list(test_results.keys())
at = [test_results[n]['accuracy']    for n in nt]
ft = [test_results[n]['f1_weighted'] for n in nt]
x  = np.arange(len(nt)); w = 0.35
fig, ax = plt.subplots(figsize=(14, 5))
b1 = ax.bar(x-w/2, at, w, label='Accuracy',    color='steelblue',  alpha=0.85)
b2 = ax.bar(x+w/2, ft, w, label='F1-Weighted', color='darkorange', alpha=0.85)
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f'{b.get_height():.3f}', ha='center', fontsize=8)
ax.axhline(1/9, color='red', ls='--', alpha=0.4, label='Chance (11.1%)')
ax.set_xticks(x); ax.set_xticklabels(nt, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.15)
ax.set_title("Test Performance – ERD/ERS (Subject-Dependent)", fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_test_performance.png", bbox_inches='tight')
plt.show()


In [ ]:
# FIG 3 – Best CM
best_name = max(test_results, key=lambda n: test_results[n]['accuracy'])
print(f"Best: {best_name}  Acc={test_results[best_name]['accuracy']:.4f}")
cm = confusion_matrix(y_test, test_results[best_name]['y_pred'])
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=LABELS_S, yticklabels=LABELS_S, linewidths=0.5)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix – {best_name}\n(ERD/ERS, Subject-Dependent)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_confusion_matrix_best.png", bbox_inches='tight')
plt.show()


In [ ]:
# FIG 4 – Classification report
rpt = classification_report(y_test, test_results[best_name]['y_pred'], output_dict=True)
rdf = pd.DataFrame(rpt).T
print(classification_report(y_test, test_results[best_name]['y_pred'],
      target_names=[SCENARIO_LABELS[i] for i in range(9)]))

fig, ax = plt.subplots(figsize=(13, 4)); ax.axis('off')
rows = [[f"S{i+1} – {SCENARIO_LABELS[i]}",
         f"{rdf.loc[str(i),'precision']:.3f}",
         f"{rdf.loc[str(i),'recall']:.3f}",
         f"{rdf.loc[str(i),'f1-score']:.3f}",
         f"{int(rdf.loc[str(i),'support'])}"]
        for i in range(9) if str(i) in rdf.index]
t = ax.table(cellText=rows,
             colLabels=['Scenario','Precision','Recall','F1-Score','Support'],
             cellLoc='center', loc='center', colColours=['#2E75B6']*5)
t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1.2, 1.7)
for j in range(5): t[0,j].set_text_props(color='white', fontweight='bold')
ax.set_title(f"Classification Report – {best_name} (ERD/ERS, Subj-Dep)",
             fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_classification_report.png", bbox_inches='tight')
plt.show()


In [ ]:
# FIG 5 – RF Feature importance
rf_imp = models["Random Forest"].named_steps['clf'].feature_importances_
fi = (pd.DataFrame({'feature': feature_cols, 'importance': rf_imp})
      .sort_values('importance', ascending=False).head(20))
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(fi['feature'], fi['importance'],
        color=plt.cm.YlOrRd(np.linspace(0.4, 0.9, 20))[::-1])
ax.set_xlabel("Importance")
ax.set_title("Top 20 Features – RF (ERD/ERS, Subject-Dependent)", fontsize=12, fontweight='bold')
ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_rf_feature_importance.png", bbox_inches='tight')
plt.show()


In [ ]:
# FIG 6 – Stacking CM
stk_key = "Stacking (RF+XGB+LGBM)"
cm_s = confusion_matrix(y_test, test_results[stk_key]['y_pred'])
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_s, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=LABELS_S, yticklabels=LABELS_S, linewidths=0.5)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix – Stacking (ERD/ERS, Subject-Dependent)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/06_stacking_confusion_matrix.png", bbox_inches='tight')
plt.show()


In [ ]:
# FIG 7 – All CMs
nm = len(test_results); ncols = 4; nrows = (nm+ncols-1)//ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows))
fig.suptitle("All Confusion Matrices – ERD/ERS (Subject-Dependent)", fontsize=13, fontweight='bold')
for idx, (name, res) in enumerate(test_results.items()):
    ax  = axes.flat[idx]
    cmi = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cmi, annot=True, fmt='d', cmap=CMAPS[idx%len(CMAPS)],
                ax=ax, xticklabels=LABELS_S, yticklabels=LABELS_S,
                linewidths=0.3, cbar=False)
    ax.set_title(f"{name}\nAcc={res['accuracy']:.3f}", fontsize=8, fontweight='bold')
    ax.tick_params(labelsize=6)
for idx in range(nm, nrows*ncols): axes.flat[idx].axis('off')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/07_all_confusion_matrices.png", bbox_inches='tight')
plt.show()


## 7. Save Results

In [ ]:
summary = pd.DataFrame([{
    'Model':        n,
    'Test_Accuracy': r['accuracy'],
    'F1_Weighted':   r['f1_weighted'],
    'CV_Mean': cv_results[n].mean() if n in cv_results else None,
    'CV_Std':  cv_results[n].std()  if n in cv_results else None,
} for n, r in test_results.items()]).sort_values('Test_Accuracy', ascending=False)

summary.to_csv(f"{OUTPUT_DIR}/model_results_summary.csv", index=False)
print(summary.to_string(index=False))
print(f"\n✅ Saved to {OUTPUT_DIR}/model_results_summary.csv")
